In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # 코랩용 경로 (구글 드라이브 내 위치)
    DATA_PATH = '/content/drive/MyDrive/legibility/vr_legibility_data.csv'
else:
    # 로컬용 경로 
    DATA_PATH = os.path.join('..', 'vr_legibility_data.csv')

In [ ]:
# 1. 데이터 로드 및 전처리
# ---------------------------------------------------------
print("데이터를 로드하고 전처리를 시작합니다...")
df = pd.read_csv(DATA_PATH)

# 다중공선성 방지: width와 height가 거의 동일하므로 width 열 제거
df = df.drop('width', axis=1)
print(df.head())

# 특성(X)과 타겟(y) 분리
X = df.drop('label', axis=1)
y = df['label']

# 학습/테스트 데이터 분할 (클래스 불균형 유지를 위해 stratify=y 적용)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

데이터를 로드하고 전처리를 시작합니다...
      width    x_coord   y_coord  label
0   74.0532  -358.5643  214.0147      0
1  216.7604  1288.1415   38.9841      0
2  349.1783   663.0387 -205.7102      0
3  292.8900   281.9659  338.8986      1
4  322.6245  -983.0987  199.5097      0


In [21]:
# 2. 파이프라인 (스케일링 + 모델) 구축
# ---------------------------------------------------------
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced', random_state=42)) # 차후 수정 필요 : 현재 비율 3.5:6.5, 불균형 해소를 위해 balanced 적용 
])

In [28]:
# 3. 커널 탐색 및 하이퍼파라미터 튜닝 (BayesSearchCV)
# ---------------------------------------------------------
kernel_configs = [
    # 공간 1: RBF 커널 전용 (C, gamma)
    ({
        'svm__kernel': Categorical(['rbf']),
        'svm__C': Real(0.1, 1000.0, prior='log-uniform'),
        'svm__gamma': Real(1e-3, 10.0, prior='log-uniform')
    }, 40),
    
    # 공간 2: Linear 커널 전용 (C)
    ({
        'svm__kernel': Categorical(['linear']),
        'svm__C': Real(1e-3, 10.0, prior='log-uniform')
    }, 20),
    
    # 공간 3: Poly 커널 전용 (C, gamma, degree)
    ({
        'svm__kernel': Categorical(['poly']),
        'svm__C': Real(0.1, 10.0, prior='log-uniform'),
        'svm__gamma': Real(1e-3, 10.0, prior='log-uniform'),
        'svm__degree': Integer(2, 4) # 2차식부터 4차식까지 탐색
    }, 50)
]

In [ ]:
# 교차 검증 객체 생성 및 BayesSearchCV 설정
best_score, best_model, best_params = -1, None, None

for space, n_iter in kernel_configs:
    bayes_search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=[space],
        n_iter=n_iter,      # 각 커널별 탐색 횟수
        cv=5,               # 5-Fold 교차 검증
        scoring='f1',       
        n_jobs=-1,          # 모든 CPU 코어 사용
        random_state=42
    )
    bayes_search.fit(X_train, y_train)
    if bayes_search.best_score_ > best_score:
        best_score = bayes_search.best_score_
        best_model = bayes_search.best_estimator_
        best_params = bayes_search.best_params_
    print(f"Completed BayesSearchCV for kernel: {space['svm__kernel'].categories[0]} with best F1: {bayes_search.best_score_:.4f}")

print(f"Best Kernel: {best_params['svm__kernel']}")
print(f"Best Params: {best_params}")
print(f"Best CV F1:  {best_score:.4f}")

Completed BayesSearchCV for kernel: rbf with best F1: 0.8003


c:\Users\psgve\AppData\Local\Programs\Python\Python313\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [10.0, np.str_('linear')] before, using random point [0.03770789360629913, 'linear']
  warnings.warn(


Completed BayesSearchCV for kernel: linear with best F1: 0.5568
Completed BayesSearchCV for kernel: poly with best F1: 0.6966
Best Kernel: rbf
Best Params: OrderedDict({'svm__C': 0.9159761565147926, 'svm__gamma': 0.7632428173841834, 'svm__kernel': 'rbf'})
Best CV F1:  0.8003


In [ ]:
# 5. 최종 평가 및 출력
# ---------------------------------------------------------
# BayesSearch가 찾은 가장 성능이 좋은 모델로 테스트 데이터를 예측합니다.
best_model = bayes_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n=========================================")
print("🎯 [BayesSearchCV 탐색 결과]")
print("=========================================")
print(f"가장 좋은 파라미터 조합: {bayes_search.best_params_}")
print(f"최고 교차 검증 점수 (F1 Macro): {bayes_search.best_score_:.4f}\n")

print("=========================================")
print("📊 [테스트 데이터 최종 평가 결과]")
print("=========================================")
print("혼동 행렬 (Confusion Matrix):")
print(confusion_matrix(y_test, y_pred))
print("\n분류 리포트 (Classification Report):")
print(classification_report(y_test, y_pred))


🎯 [GridSearchCV 탐색 결과]
가장 좋은 파라미터 조합: {'svm__C': 1, 'svm__gamma': 10, 'svm__kernel': 'rbf'}
최고 교차 검증 점수 (F1 Macro): 0.5224

📊 [테스트 데이터 최종 평가 결과]
혼동 행렬 (Confusion Matrix):
[[ 2  8]
 [10 80]]

분류 리포트 (Classification Report):
              precision    recall  f1-score   support

           0       0.17      0.20      0.18        10
           1       0.91      0.89      0.90        90

    accuracy                           0.82       100
   macro avg       0.54      0.54      0.54       100
weighted avg       0.83      0.82      0.83       100

